# Spark MLlib - End-to-End ML Pipeline
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/07_BigData_Spark/pyspark_mllib_pipeline.ipynb)

Spark MLlib scales scikit-learn-style workflows to cluster-sized data using DataFrames and Pipelines: every preprocessing step becomes a stage that fits/transforms consistently in production.

Runs fully local (driver JVM) - no cluster needed.

In [ ]:
!pip install -q pyspark

## 1. Session + synthetic churn-like data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("mllib-pipeline")
         .getOrCreate())
print("spark version:", spark.version)

n = 2000
df = (spark.range(n)
      .withColumn("tenure_months", (F.rand(seed=1) * 60).cast("int"))
      .withColumn("monthly_charges", 20 + F.rand(seed=2) * 70)
      .withColumn("contract", F.when(F.rand(seed=3) > 0.6, "annual")
                                 .when(F.rand(seed=4) > 0.4, "monthly")
                                 .otherwise("biannual"))
      .withColumn("churn", F.when((F.col("contract") == "monthly") &
                                  (F.col("tenure_months") < 12), 1.0)
                             .otherwise(F.when(F.rand(seed=5) < 0.15, 1.0)
                                        .otherwise(0.0))))
df.show(5)
df.groupBy("contract").agg(F.avg("churn").alias("churn_rate")).show()

## Pipeline stages
| Stage | Role |
|---|---|
| `StringIndexer` | category -> index |
| `VectorAssembler` | feature columns -> single `features` vector |
| classifier | learns from the vector |

## 2. Assemble the Pipeline

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

indexer = StringIndexer(inputCol="contract", outputCol="contract_idx")
assembler = VectorAssembler(
    inputCols=["tenure_months", "monthly_charges", "contract_idx"],
    outputCol="features")
rf = RandomForestClassifier(labelCol="churn", featuresCol="features",
                            numTrees=50, seed=42)

pipeline = Pipeline(stages=[indexer, assembler, rf])
train, test = df.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(train)          # fits every stage in order
pred = model.transform(test)
pred.select("churn", "probability", "prediction").show(5, truncate=False)

## 3. Evaluate

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc = BinaryClassificationEvaluator(labelCol="churn",
                                    metricName="areaUnderROC").evaluate(pred)
acc = MulticlassClassificationEvaluator(labelCol="churn", predictionCol="prediction",
                                        metricName="accuracy").evaluate(pred)
print(f"AUC = {auc:.4f}   accuracy = {acc:.4f}")

rf_model = model.stages[-1]
print("feature importances:",
        dict(zip(assembler.getInputCols(),
                 [round(v, 3) for v in rf_model.featureImportances.toArray()])))

## 4. Hyperparameter tuning with CrossValidator

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

grid = (ParamGridBuilder()
        .addGrid(rf.numTrees, [20, 50])
        .addGrid(rf.maxDepth, [3, 6])
        .build())

cv = CrossValidator(estimator=pipeline,
                    estimatorParamMaps=grid,
                    evaluator=BinaryClassificationEvaluator(labelCol="churn"),
                    numFolds=3, seed=42)
cv_model = cv.fit(train)
best = cv_model.bestModel.stages[-1]
print(f"best numTrees={best.getNumTrees} maxDepth={best.getOrDefault(best.maxDepth)}")
print(f"CV AUC = {max(cv_model.avgMetrics):.4f}")

**Why pipelines matter:** the SAME fitted object handles raw input at serving time - categories unseen in training map correctly, vector order never drifts. Save/load with `model.write().overwrite().save("path")`.

Next-level: connect to real storage (`spark.read.parquet/s3/jdbc`) and scale `master("local[*]")` out to a cluster URL - code unchanged.